# Quantum Funding Chart — Smooth World Edition

Geographic bubble chart of estimated public quantum-technology funding (2026 snapshot).

This notebook uses a **smooth gray world silhouette** as a quiet base layer so the funding bubbles stay the focus. The silhouette is pure Matplotlib (no Cartopy, GeoPandas, or external map files). Bubble area scales with the annual or annualized figure. China is shown but flagged as a cumulative estimate and excluded from annual totals.

**Run all cells from the top** in a fresh Colab runtime. The world layer is defined in section 3A and drawn with `draw_smooth_world(ax)`.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_DIR = Path("/content/mit-prox-rd")
REPO_URL = "https://github.com/qimono-repos/mit-prox-rd.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "trunk", "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", "trunk"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", "trunk", "FETCH_HEAD"], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
print("CSV exists:", Path("assets/quantum_funding_updated.csv").exists())

## 1. Imports and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Polygon
import numpy as np
from pathlib import Path

csv_path = Path("assets/quantum_funding_updated.csv")
if not csv_path.is_file():
    raise FileNotFoundError(f"CSV not found: {csv_path.resolve()}")

df = pd.read_csv(csv_path, encoding="utf-8-sig")
print(f"Loaded {len(df)} records from {csv_path}.")
display(df.head())

## 2. Configuration

Country values (annual or annualized) and approximate geographic coordinates.

In [ ]:
values = {
    "United States": 1006,
    "Canada": 38.5,
    "Brazil": 5.5,
    "United Kingdom": 330,
    "Germany": 848,
    "Netherlands": 85,
    "Switzerland": 23,
    "Austria": 24,
    "Italy": 36,
    "India": 90,
    "South Korea": 140,
    "Japan": 241,
    "Singapore": 44,
    "Australia": None,
    "China": 15000,
}

pos = {
    "Canada": (-105, 57),
    "United States": (-99, 37),
    "Brazil": (-52, -10),
    "United Kingdom": (-3, 54),
    "Netherlands": (5, 53),
    "Germany": (11, 51),
    "Switzerland": (8, 47),
    "Austria": (14, 47),
    "Italy": (12, 42),
    "India": (79, 22),
    "South Korea": (128, 36),
    "Japan": (139, 37),
    "Singapore": (104, 1.3),
    "Australia": (134, -25),
    "China": (105, 35),
}

## 3. Smooth World Layer

Reusable pure-Matplotlib world silhouette. Coastlines are hand-tuned polygons, then smoothed with Chaikin subdivision so they look closer to a clean atlas outline (inspired by a solid black world silhouette).

In [ ]:
WORLD_FACE = "#e6e9ec"
WORLD_EDGE = "#c5ccd3"

WORLD_CONTINENTS = {
    "north_america": [
        (-168,66),(-162,70),(-153,70),(-146,66),(-139,61),(-132,58),(-128,52),
        (-125,48),(-123,42),(-117,33),(-111,30),(-105,26),(-98,22),(-91,18),
        (-86,12),(-82,8),(-78,8),(-80,15),(-76,22),(-80,28),(-76,34),
        (-70,42),(-64,47),(-59,53),(-61,58),(-72,62),(-85,65),(-99,70),
        (-117,73),(-135,72),(-151,70),(-161,68)
    ],
    "greenland": [(-53,83),(-44,84),(-34,81),(-25,76),(-29,70),(-38,64),(-47,59),(-55,61),(-61,68),(-59,76)],
    "south_america": [
        (-81,12),(-76,9),(-71,7),(-65,3),(-59,0),(-52,-4),(-46,-9),
        (-40,-16),(-42,-23),(-47,-31),(-52,-39),(-57,-47),(-63,-55),
        (-68,-51),(-71,-41),(-74,-28),(-78,-13),(-80,0)
    ],
    "eurasia": [
        (-10,36),(-9,43),(-5,49),(1,53),(7,57),(13,60),(20,62),(27,67),
        (37,70),(47,67),(56,64),(66,62),(76,60),(87,58),(98,56),(110,53),
        (122,51),(135,49),(148,46),(160,42),(169,35),(166,29),(157,23),
        (148,18),(140,11),(132,6),(124,8),(117,14),(109,19),(101,22),
        (93,20),(85,24),(77,28),(69,33),(60,38),(52,37),(45,33),(38,29),
        (32,31),(27,37),(20,39),(14,36),(8,36),(3,40),(-2,39)
    ],
    "africa": [
        (-17,36),(-5,37),(8,35),(19,32),(28,27),(34,19),(38,9),(43,11),
        (50,11),(49,3),(42,-2),(38,-12),(32,-22),(25,-33),(17,-35),
        (10,-29),(4,-18),(-1,-5),(-7,8),(-13,20)
    ],
    "india": [(68,27),(78,30),(88,23),(84,13),(77,7),(73,16)],
    "japan": [(130,45),(136,43),(142,39),(145,35),(141,32),(136,34),(132,39)],
    "australia": [
        (113,-11),(115,-19),(121,-24),(129,-25),(137,-17),(145,-15),
        (153,-25),(149,-37),(139,-43),(128,-40),(117,-34),(114,-24)
    ],
    "new_zealand": [(172,-35),(178,-41),(174,-46),(169,-42)],
    "madagascar": [(49,-12),(51,-17),(49,-25),(46,-21)],
}

def _smooth_closed(points, iterations=3):
    pts = [(float(x), float(y)) for x, y in points]
    for _ in range(iterations):
        refined = []
        for i, p in enumerate(pts):
            q = pts[(i + 1) % len(pts)]
            refined.append((0.75*p[0] + 0.25*q[0], 0.75*p[1] + 0.25*q[1]))
            refined.append((0.25*p[0] + 0.75*q[0], 0.25*p[1] + 0.75*q[1]))
        pts = refined
    return pts

def draw_smooth_world(ax, zorder=1):
    """Draw a smooth gray world silhouette on a Matplotlib axis."""
    for name, points in WORLD_CONTINENTS.items():
        smooth = _smooth_closed(points, iterations=3)
        patch = Polygon(
            smooth, closed=True,
            facecolor=WORLD_FACE,
            edgecolor=WORLD_EDGE,
            linewidth=0.7,
            joinstyle="round",
            antialiased=True,
            zorder=zorder,
        )
        ax.add_patch(patch)
    return ax

## 4. Generate the Chart

In [ ]:
fig = plt.figure(figsize=(18, 11), dpi=160, facecolor="#fbfaf7")
ax = fig.add_axes([0.04, 0.18, 0.92, 0.64])
ax.set_facecolor("#fbfaf7")
ax.set_xlim(-175, 185)
ax.set_ylim(-58, 82)
ax.axis("off")

# Smooth world base
draw_smooth_world(ax, zorder=1)

# Bubbles
max_v = max(v for v in values.values() if v is not None and v < 5000)
for country, v in values.items():
    if country not in pos:
        continue
    x, y = pos[country]
    if v is None:
        ax.scatter([x], [y], s=90, facecolors="none", edgecolors="#6b7280", linewidths=1.6, zorder=4)
        continue
    if country == "China":::
        radius, color, edge = 7.2, "#f6e8d4", "#b96b26"
    else:
        radius = 1.8 + 8.2 * np.sqrt(v / max_v)
        color, edge = "#69c7d7", "#00a9c0"
    ax.add_patch(Circle((x, y), radius=radius, facecolor=color, edgecolor=edge,
                        linewidth=1.4, alpha=0.78, zorder=3))
    if v >= 200 and country != "China":::
        ax.text(x, y, f"{v:,.0f}", ha="center", va="center", fontsize=9.5, color="#17333a", weight="bold", zorder=5)
        ax.text(x, y + 4.2, country, ha="center", va="center", fontsize=8.5, color="#17333a", weight="bold", zorder=6)
    elif country == "China":::
        ax.text(x, y, "≈15,000", ha="center", va="center", fontsize=10, color="#7d4519", weight="bold", zorder=5)
        ax.text(x, y + 5.2, "China\n(cumulative)", ha="center", va="center", fontsize=8, color="#7d4519", weight="bold", zorder=6)

# Titles
fig.text(0.04, 0.93, "Quantum funding — 2026 snapshot", fontsize=24, weight="bold", color="#1a1a1a")
fig.text(0.04, 0.885, "Estimated annual or annualized public funding (USD millions). Bubble area scales with the figure.", fontsize=12, color="#444")

# China caveat
fig.text(0.72, 0.22, "CHINA — SPECIAL EXCEPTION\n≈ US$15B cumulative estimate\nthrough ~2024 • high uncertainty\nNot comparable to annual figures",
         ha="center", va="center", fontsize=9, color="#7d4519",
         bbox=dict(boxstyle="round,pad=0.55", facecolor="#f6e8d4", edgecolor="#b96b26", linewidth=1.2))

fig.text(0.04, 0.06, "US = FY2024 enacted. Multi-year programs are annualized estimates. Australia: outlined marker only.", fontsize=8.5, color="#555")
fig.text(0.04, 0.04, "Sources: project CSV + linked national programs. See REFERENCES.md.", fontsize=8, color="#555")

plt.show()

## 5. Notes

- U.S. figure is FY2024 enacted.
- Multi-year programs are annualized estimates, not single-year appropriations.
- China is a cumulative estimate with high uncertainty and is not on the same scale.
- See `REFERENCES.md` for the full source list.
- This is an editorial geographic illustration, not a GIS basemap.